# NIST Photovoltaic Data: Inspection, Feature Engineering and Fault Labeling

This notebook presents the methodology stage by stage for one selected Canopy or Ground dataset. It follows the structure of the original annual notebooks, using individual shared functions from `scripts/` for the calculations. Change the configuration to repeat the analysis across 2015–2018.

The compact `Site_Year_Labeling.ipynb` remains a separate batch runner. Here each transformation is followed by a report, table or plot so its effect can be examined and discussed in the diploma thesis. Findings are calculated when run; observations from one year are not assumed to hold for another.

## Notebook structure

1. Data Inspection
2. Feature Selection: secondary features, redundant features, faulty sensors, correlation
3. Feature Engineering: missing values, measurement bounds, operating mask
4. Expected vs Actual Power Analysis (Performance Ratio)
5. String Level Faults
6. Inverter Level Faults
7. Final validation, summaries and export

The three label families are rule-based supervision signals, not independently verified failure diagnoses. They remain separate for later analysis.


## Colab setup

Run in Google Colab, from top to bottom, after `Data_Overview.ipynb` has created the merged annual CSVs. The public repository supplies code only; datasets and exported results stay on Google Drive.

An existing checkout is reused within a runtime. Restart the runtime to clone updated code. When changing site/year, rerun from the configuration cell onward. Reapplying individual transformations to an already processed DataFrame may fail because columns were removed or features already exist.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/dulkusim/pv-timeseries-fault-forecasting.git"
repo_path = Path("/content/pv-timeseries-fault-forecasting")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(repo_path)],
        check=True,
    )

labeling_dir = repo_path / "notebooks" / "Data Labeling"
if not (labeling_dir / "scripts" / "labeling.py").is_file():
    raise FileNotFoundError(f"Shared scripts not found in {labeling_dir}")
if str(labeling_dir) not in sys.path:
    sys.path.insert(0, str(labeling_dir))

import calendar
import json
from dataclasses import asdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Import each stage explicitly so its responsibility remains visible.
from scripts.config import CANOPY, GROUND, POA_MIN_OPERATING, POA_MIN_PR
from scripts.io import get_data_root, load_site_year, validate_site_year, save_stage
from scripts.feature_selection import (
    SECONDARY_FEATURES, REDUNDANT_FEATURES, CORRELATION_COLUMNS,
    drop_features, rtd_report, detect_faulty_rtds, drop_legacy_rtds,
    correlation_report,
)
from scripts.cleaning import missing_report, apply_missingness_policy
from scripts.operating import operating_mask, poa_fused, module_temp, pr_evaluable
from scripts.performance import expected_power, performance_features, label_power
from scripts.strings import (
    CURRENT_MIN, CURRENT_MAX, POWER_MIN, POWER_MAX,
    POA_MIN_STRING, STRING_REL_DEV_THRESHOLD, OPEN_CIRCUIT_RATIO,
    current_columns, power_columns, clean_string_bounds,
    string_aggregates, string_voltage_proxy, string_normalized,
    fit_string_baselines, apply_string_baselines, label_string,
)
from scripts.inverter import (
    FAULT_FLAG_COLUMNS, ORIGINAL_INVERTER_POLICY, REVISED_INVERTER_POLICY,
    inverter_features, label_inverter,
)
from scripts.labeling import summarize


## Google Drive and site configuration

`config.py` holds physical ratings, sensor selections, RTD removal choices and core circuits. Select `SITE_NAME` and `YEAR` below; the same notebook covers both sites and all four years.

Inputs come from `MyDrive/NIST_processed/`. Final CSVs go to `MyDrive/NIST_Labels/<site>_<year>.csv`.

Optional intermediate checkpoints use `io.save_stage` under `MyDrive/NIST_labeling_stages/`. They default to off to avoid several large CSV copies. That helper always rejects existing checkpoints; `OVERWRITE_FINAL` applies only to final exports.

The original inverter policy is selected explicitly to retain the annual notebooks' rules. The revised policy remains an option, explained in the inverter section.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.environ["DATA_ROOT"] = "/content/drive/MyDrive"
root = get_data_root()

SITE_NAME = "canopy"
YEAR = 2015
SAVE_CHECKPOINTS = False
OVERWRITE_FINAL = False
INVERTER_POLICY = REVISED_INVERTER_POLICY

sites = {"canopy": CANOPY, "ground": GROUND}
if SITE_NAME not in sites or YEAR not in (2015, 2016, 2017, 2018):
    raise ValueError("Choose canopy/ground and a year from 2015 through 2018")
site = sites[SITE_NAME]

output_dir = root / "NIST_Labels"
final_path = output_dir / f"{site.name}_{YEAR}.csv"
summary_path = output_dir / f"{site.name}_{YEAR}_label_summary.csv"
audit_path = output_dir / f"{site.name}_{YEAR}_stage_audit.csv"
metadata_path = output_dir / f"{site.name}_{YEAR}_methodology.json"
print(f"Selected dataset: {site.name}, {YEAR}")
display(pd.Series(asdict(site), name="configuration").to_frame())
print(f"Inverter policy: {INVERTER_POLICY}")


## Reporting utilities

The reporting functions are defined in the shared `scripts/reporting.py` module, keeping their implementations outside the notebook. This cell imports them and initializes the stage audit for the selected site/year.

`append_features` checks timestamp alignment and duplicate feature names. `report_stage` records dataset dimensions and optionally saves a checkpoint. `label_distribution` shows counts and percentages of all observations.

The notebook supplies its configuration to `report_stage` once using `partial`; the script does not depend on notebook global variables. Only a single working DataFrame is retained between stages, although calculations can still need substantial temporary Colab memory.


In [ ]:
from functools import partial
from scripts.reporting import append_features, label_distribution, report_stage as record_stage

stage_records = []

# Bind the current run's reporting settings; later stages use report_stage directly.
report_stage = partial(
    record_stage,
    records=stage_records,
    site=site,
    year=YEAR,
    save_checkpoints=SAVE_CHECKPOINTS,
)


# Data Inspection

The merged CSV is loaded through `load_site_year`. This helper parses `TIMESTAMP`, sets it as the index, sorts observations chronologically and rejects empty datasets, missing timestamps or observations from the wrong year. This prevents accidental reuse of another year's file.

Inspect records, types, ranges, shape and memory usage before removing features. Times retain their recorded local standard time; no timezone conversion is introduced.


In [ ]:
# Load the original merged measurements; no engineered features are expected yet.
df = load_site_year(site, YEAR)
original_rows = len(df)
original_index = df.index.copy()
report_stage("Raw measurements", df)
display(df.head())


In [ ]:
# Feature types, nonmissing counts and memory usage.
df.info()


In [ ]:
# Summary statistics for the original measurements.
df.describe()


In [ ]:
# Number of observations and measurement columns.
df.shape


In [ ]:
# Total number of measurement cells.
df.size


In [ ]:
# List the available measurements and their types.
display(pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).to_numpy(),
}))


## Dataset Column Descriptions

| Measurement group | Columns | Meaning |
|---|---|---|
| Time and logging | TIMESTAMP, RECORD, CR1000Temp_C_Avg | Recorded time, record number and logger temperature |
| Irradiance | Pyra*_Wm2_Avg, RefCell*_Wm2_Avg | Pyranometer/reference-cell irradiance (W/m²); sensor choice depends on site |
| Temperature | AmbTemp_C_Avg, RTD_C_Avg_1–9, RTD_C_Avg_10 | Ambient, module backsheet and inverter interior temperatures (°C) |
| Inverter power | InvPAC_kW_Avg, InvPDC_kW_Avg, InvEtot_kWh_Max | AC/DC power (kW) and cumulative energy (kWh) |
| Inverter electrical measurements | InvVa/b/c_Avg, InvIa/b/c_Avg, InvVDCin_Avg, InvIDCin_Avg, InvVPVin_Avg, InvVDVoltage_V_Avg, InvFreq_Avg | Phase measurements, DC voltages/current and frequency |
| Status and reported faults | InvOpStatus_Avg, Inv*Fault_Max, InvSystemWarn_Max, InvPVMStatus_Max | Operating status, fault/warning codes and monitoring status |
| Shunt-monitored circuits | ShuntCurrent_A_Avg_1–7, ShuntPDC_kW_Avg_1–7, ShuntEtot_kWh_Max_1–7 | Circuit current (A), power (kW) and cumulative energy (kWh) |
| Independent AC meter | PwrMtr* | Phase measurements, frequency, power, energy and distortion |
| Additional environment | SEWS*, Wind* | Integrator environmental sensors, wind measurements and diagnostics |
| DAS maintenance | DoorOpen_Min, Battery*, Load_A_Avg, Chg*, CkBatt_Max, Qloss_Ah_Max | Enclosure, battery and charging diagnostics |

The seven channels called “strings” in the scripts represent shunt-monitored output circuits/combiner boxes, rather than necessarily seven individual module strings. Their configured sensor roles are displayed below.


In [ ]:
display(pd.DataFrame([
    {"purpose": "Operating mask", "columns": site.op_mask_columns, "aggregation": site.op_mask_agg},
    {"purpose": "Expected-power POA", "columns": site.poa_columns, "aggregation": site.poa_agg},
]))


## Timestamp Continuity and Annual Coverage

A complete one-minute year contains 525,600 observations, or 527,040 in a leap year. Compare annual and monthly row counts with these calendar expectations, and inspect unique timestamps, duplicates and steps that differ from one minute.

The recorded timezone is preserved. Continuity is calculated directly from consecutive timestamps without comparing them with a separate timezone-naive annual grid. Counts alone do not prove completeness: duplicates can compensate for missing observations, and counts do not establish the exact annual endpoints. The observed timestamp range is shown for that reason.

The scripts use row-based rolling windows. Five, fifteen and sixty rows represent those minute durations only under regular one-minute sampling. These diagnostics do not fill gaps or remove duplicates; irregularities should inform the later analysis.


In [ ]:
# Use calendar counts for annual expectations and recorded timestamps for continuity.
expected_minutes = (366 if calendar.isleap(YEAR) else 365) * 24 * 60
time_steps = df.index.to_series().diff().dropna()
coverage = pd.Series({
    "expected_minutes": expected_minutes,
    "observed_rows": len(df),
    "unique_timestamps": df.index.nunique(),
    "duplicate_timestamps": int(df.index.duplicated().sum()),
    "steps_different_from_one_minute": int(time_steps.ne(pd.Timedelta(minutes=1)).sum()),
}, name="coverage")
display(coverage.to_frame())
print(f"Observed range: {df.index.min()} to {df.index.max()}")

monthly_rows = df.groupby(df.index.month).size().reindex(range(1, 13), fill_value=0)
monthly_expected = pd.Series({m: calendar.monthrange(YEAR, m)[1] * 24 * 60 for m in range(1, 13)})
display(pd.DataFrame({"observed_rows": monthly_rows, "expected_minutes": monthly_expected}))


# Feature Selection

Non-essential, redundant and historically unreliable features are removed before feature engineering. Each removal is shown separately so its purpose and effect remain visible. Column groups are maintained in `feature_selection.py`, and sensor policies in `config.py`.

## Secondary Feature Removal

The first group contains datalogger/maintenance variables, battery and charging measurements, secondary wind diagnostics and cumulative circuit energy counters. The established feature policy excludes these to focus on electrical behavior and environmental inputs used by the rules. This selection does not establish that the excluded variables could never help another predictive task.


In [ ]:
# Show the shared policy instead of repeating its list in every annual notebook.
display(pd.DataFrame({"secondary_feature_removed": SECONDARY_FEATURES}))
df = drop_features(df, groups=("secondary",))
report_stage("Secondary features removed", df)


## Redundant Feature Removal

The second group includes secondary meter frequency statistics, reactive/apparent power and energy counters, harmonic distortion measurements and integrator environmental sensors. The policy removes those columns while retaining the principal inverter and independent meter measurements for subsequent analysis.


In [ ]:
display(pd.DataFrame({"redundant_feature_removed": REDUNDANT_FEATURES}))
df = drop_features(df, groups=("redundant",))
report_stage("Redundant features removed", df)


## Faulty Sensor Detection

The nine module RTDs are inspected by comparing annual mean temperatures. `detect_faulty_rtds` flags sensors whose mean differs by more than 10 °C from the median sensor mean. This is a screening diagnostic: an unusual mean motivates inspection, while a plausible mean does not prove year-round reliability.

Removal follows the established site policy to preserve consistency across annual datasets:

- Canopy removes RTDs 3, 5, 6 and 7; later module temperature uses 1, 2, 4, 8 and 9.
- Ground removes RTDs 2, 3 and 8; later module temperature uses 1, 4, 5, 6, 7 and 9.

The table and plot provide evidence for the selected year. The diagnostic candidates do not automatically replace the configured removal policy.


In [ ]:
rtd_means = rtd_report(df)
suspect_rtds = detect_faulty_rtds(df, tol=10.0)
rtd_means["diagnostic_candidate"] = rtd_means.index.isin(suspect_rtds)
rtd_means["removed_by_site_policy"] = rtd_means.index.isin(
    [f"RTD_C_Avg_{i}" for i in site.rtd_drop]
)
display(rtd_means)
ax = rtd_means["mean_c"].plot.bar(
    figsize=(10, 3), ylabel="Annual mean temperature (°C)"
)
ax.set_title(f"{site.name.title()} {YEAR}: module RTD comparison")
plt.tight_layout()
plt.show()


In [ ]:
# Retain the same module sensor set across years for this installation.
df = drop_legacy_rtds(df, site)
report_stage("Site-specific RTDs removed", df)


## Correlation Analysis for Feature Redundancy

A Pearson correlation matrix compares retained inverter and meter electrical measurements. Inverter/meter phase currents and AC power may capture overlapping behavior while still describing different measurement noise or grid-side effects.

`correlation_report` lists pairs with absolute correlation at least 0.9. The full matrix and heatmap are also shown. This is a diagnostic rather than an automatic elimination rule: these selected electrical columns remain available for labeling and later model-based selection. Discuss the calculated values rather than assuming the correlations match a previous year.


In [ ]:
correlations = df.loc[:, list(CORRELATION_COLUMNS)].corr()
display(correlations.round(3))
display(correlation_report(df, threshold=0.9))

fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(correlations, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlations)), correlations.columns, rotation=90)
ax.set_yticks(range(len(correlations)), correlations.index)
ax.set_title(f"{site.name.title()} {YEAR}: electrical feature correlations")
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()
report_stage("Feature selection complete", df, checkpoint="features")


# Feature Engineering

## Missing Values

`missing_report` lists missing counts and fractions. The established cross-site policy removes `WindSpeedAve_ms`; it does **not** remove every column above a percentage threshold. Critical electrical measurements remain even when incomplete.

The `SEWS*` environmental columns were already removed in the redundant group. No general interpolation or row deletion is introduced. Subsequent masks and label precedence determine how incomplete observations are categorized; not every missing input results in a missing-data label.


In [ ]:
missing_before = missing_report(df)
display(missing_before.assign(missing_percent=100 * missing_before["missing_fraction"]))

# Preserve the shared column policy for both sites and all years.
df = apply_missingness_policy(df)
display(missing_report(df))
report_stage("Missingness policy applied", df)


## Physical Bounds for String Measurements

The shared pipeline replaces currents outside 0–150 A and powers outside 0–50 kW with missing values for each of the seven circuits. Rows are retained. The report distinguishes original missing values from measurements rejected by these bounds.

Cleaning occurs here before operating/performance features to match the shared pipeline order. Its effect is revisited in the string section.


In [ ]:
string_current_cols = current_columns(site.n_strings)
string_power_cols = power_columns(site.n_strings)
string_measurements = string_current_cols + string_power_cols
bounds_report = pd.DataFrame({"missing_before": df[string_measurements].isna().sum()})
print(f"Allowed current: {CURRENT_MIN}–{CURRENT_MAX} A; power: {POWER_MIN}–{POWER_MAX} kW")
df = clean_string_bounds(df)
bounds_report["missing_after"] = df[string_measurements].isna().sum()
bounds_report["rejected_by_bounds"] = bounds_report["missing_after"] - bounds_report["missing_before"]
display(bounds_report)
report_stage("Bounds cleaning complete", df, checkpoint="cleaned")


## Operating Mask Creation

`Op_Mask` distinguishes sufficient irradiance from low-irradiance/nighttime conditions:

- Canopy uses the maximum of `Pyra1_Wm2_Avg` and `Pyra3_Wm2_Avg` to account for the two orientations.
- Ground uses `Pyra2_Wm2_Avg` alone.

The mask equals 1 when the configured irradiance is **strictly greater than 100 W/m²**, and 0 otherwise. It describes expected operation from irradiance, not confirmation that the inverter is producing.

The helper follows pandas aggregation/comparison behavior for missing readings. In particular, a missing result fails the threshold comparison and becomes 0; missing irradiance is not separately labeled at this stage.


In [ ]:
df["Op_Mask"] = operating_mask(df, site)
print(f"Operating irradiance threshold: > {POA_MIN_OPERATING} W/m²")
display(label_distribution(df["Op_Mask"]))

operating_rows = int(df["Op_Mask"].sum())
print(f"Operating observations: {operating_rows:,} ({100 * operating_rows / len(df):.2f}% of rows)")
print(f"Equivalent sampled hours at one-minute spacing: {operating_rows / 60:,.2f}")
ax = df["Op_Mask"].groupby(df.index.month).mean().mul(100).plot.bar(
    figsize=(10, 3), ylabel="Operating observations (%)", xlabel="Month"
)
ax.set_title(f"{site.name.title()} {YEAR}: monthly operating fraction")
plt.tight_layout()
plt.show()
report_stage("Operating mask created", df, checkpoint="operating")


# Expected vs Actual Power Analysis (Performance Ratio)

Physics-based features compare measured inverter AC output with an irradiance- and temperature-based power reference. Inputs are site DC rating, reference-cell POA, retained module RTDs, temperature coefficient and inverter AC ceiling.

This is the project's reference model, not a full simulation of all conversion/environmental losses. The ratio thresholds provide heuristic supervision and do not independently establish a particular physical fault.

## POA Irradiance, Module Temperature and PR Evaluation Mask

Canopy `POA_fused` is the **mean** of the two reference cells; Ground uses `RefCell1_Wm2_Avg`. This differs from the pyranometer maximum used for Canopy's operating mask. Module temperature is the row-wise median of configured retained RTDs.

`PR_evaluable` requires operating mask 1, POA at least 200 W/m² and inverter status strictly above 0 and at most 5. The higher irradiance threshold reduces sensitivity to ratios at weak illumination. Missing module temperature or AC output is handled later by the power label function rather than this mask.


In [ ]:
# Keep POA fusion and module RTD selection specific to the chosen site.
df["POA_fused"] = poa_fused(df, site)
df["module_temp"] = module_temp(df, site)
df["PR_evaluable"] = pr_evaluable(df, site)
print(f"PR irradiance threshold: >= {POA_MIN_PR} W/m²")
display(df[["POA_fused", "module_temp"]].describe().T)
display(label_distribution(df["PR_evaluable"]))
display(pd.crosstab(df["Op_Mask"], df["PR_evaluable"], margins=True))


## Temperature-Corrected Expected Power

The expected DC reference is

$$P_{expected,DC}=P_{rated,DC}\frac{G_{POA}}{1000}\left[1+\gamma(T_{module}-T_{STC})\right].$$

Configuration uses 243 kW DC for Canopy, 271 kW DC for Ground, $\gamma=-0.00485$ per °C and $T_{STC}=25$ °C. The AC reference is capped at the 260 kW inverter ceiling:

$$P_{expected,AC}=\min(P_{expected,DC},P_{AC,max}).$$

The cap limits the reference when DC potential exceeds inverter capacity; it does not explicitly estimate conversion efficiency. The helper applies an upper cap only and preserves the remaining calculated values, including missing values.


In [ ]:
df = append_features(df, expected_power(df, site))
display(df[["InvPAC_kW_Avg", "P_expected_DC_kW", "P_expected_AC_kW"]].describe().T)


## Performance Ratio and Energy-Weighted Rolling Features

Instantaneous PR is $P_{actual,AC}/P_{expected,AC}$, with zero reference power replaced by a missing denominator. The weighted rolling ratio uses evaluable observations:

$$PR_{weighted,w}=\frac{\sum_w P_{actual,AC}}{\sum_w P_{expected,AC}}.$$

This ratio of sums reduces the influence of low-power observations compared with an unweighted mean of individual ratios. The helper produces 5-, 15- and 60-row windows with at least 3, 8 and 30 valid observations respectively. These represent minute windows under regular one-minute sampling.

The 15-row ratio drives labels. Other windows, instantaneous PR, rolling standard deviation (15 rows, minimum 8) and power deficit (expected minus actual) provide diagnostic/model features. The standard deviation uses instantaneous PR without the evaluability mask, matching the script.

Selected power/performance columns are rounded to three decimals **after calculation and before labeling**, as in the shared pipeline. This can affect threshold-adjacent observations.


In [ ]:
df = append_features(df, performance_features(df))
round_columns = [
    "module_temp", "P_expected_DC_kW", "P_expected_AC_kW", "PR",
    "PR_roll_5min_weighted", "PR_roll_15min_weighted", "PR_roll_60min_weighted",
    "PR_roll_std_15min", "power_deficit_kW",
]
df[round_columns] = df[round_columns].round(3)
display(df.loc[df["PR_evaluable"].eq(1), round_columns].describe().T)


## Expected vs Actual Power: Visual Inspection

Daily means expose annual variation without drawing every minute. They include all observations and are descriptive averages, not daily energy totals. Gaps can affect their interpretation.

A selected day shows one-minute power and PR. The first day with a PR-evaluable observation is chosen automatically; change `PLOT_DAY` to inspect an event. If no observations are PR-evaluable, the first recorded day is shown.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
df[["InvPAC_kW_Avg", "P_expected_AC_kW"]].resample("D").mean().plot(ax=ax)
ax.set(ylabel="Daily mean power (kW)", title=f"{site.name.title()} {YEAR}: expected and measured AC power")
plt.tight_layout()
plt.show()

evaluable_times = df.index[df["PR_evaluable"].eq(1)]
PLOT_DAY = (evaluable_times[0] if len(evaluable_times) else df.index[0]).strftime("%Y-%m-%d")
day = df.loc[PLOT_DAY, ["InvPAC_kW_Avg", "P_expected_AC_kW", "PR", "PR_roll_15min_weighted"]]
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
day[["InvPAC_kW_Avg", "P_expected_AC_kW"]].plot(ax=axes[0])
axes[0].set(ylabel="Power (kW)", title=f"{site.name.title()}: {PLOT_DAY}")
day[["PR", "PR_roll_15min_weighted"]].plot(ax=axes[1])
axes[1].set_ylabel("Performance ratio")
plt.tight_layout()
plt.show()


## Physics-Based Fault Labeling

`label_power` evaluates these conditions in order; the **first matching condition wins**.

| Priority | Label | Rule |
|---|---|---|
| 1 | NOT_EVALUABLE | PR_evaluable = 0 |
| 2 | MISSING_DATA | Expected/measured AC power missing, or 15-row weighted PR missing for an evaluable observation |
| 3 | CRITICAL_UNDERPERFORMANCE | Evaluable and $0 \le PR_{15} < 0.50$ |
| 4 | SEVERE_UNDERPERFORMANCE | Evaluable and $0.50 \le PR_{15} < 0.70$ |
| 5 | UNDERPERFORMANCE | Evaluable and $0.70 \le PR_{15} < 0.85$ |
| 6 | SUSPICIOUSLY_HIGH | Evaluable and $PR_{15} > 1.05$ |
| 7 | NEGATIVE_AC_POWER | Evaluable and instantaneous AC power negative, if no earlier condition matched |
| Default | NORMAL | No preceding condition matched |

The bands distinguish degrees of departure from the reference. They can motivate inspection for shading, soiling, electrical problems or measurement disagreement, but do not uniquely identify those causes. The normal PR band includes both 0.85 and 1.05 when no other rule takes precedence.

Precedence matters: missing status/irradiance can make an observation non-evaluable before missing-data checks. Negative instantaneous power does not override an earlier PR category.


In [ ]:
# A: Power-performance labels from the shared rules.
df["label_power"] = label_power(df, missing_label="MISSING_DATA")
display(label_distribution(df["label_power"]))
ax = df["label_power"].value_counts().plot.barh(figsize=(10, 4), xlabel="Observations")
ax.set_title("A: Power-performance labels")
plt.tight_layout()
plt.show()
report_stage("Performance features and power labels", df, checkpoint="performance")


# String Level Faults

String analysis distinguishes disagreement between circuits from whole-system power loss. It uses the seven shunt current/power channels after bounds cleaning. The sequence follows the original notebooks: aggregates, per-circuit deviations, voltage proxies, irradiance normalization, individual baselines, rolling deviations and labels.

## 1. String Current and Power Aggregates

`string_aggregates` calculates current mean, standard deviation, minimum/maximum, coefficient of variation ($CV=\sigma/\mu$), range relative to mean and per-circuit standardized deviations. It also calculates power mean, standard deviation, total and CV, and a 5-row current-CV average with at least 3 observations.

Large spread indicates disagreement, but structural differences can cause persistent spread without faults. `P_string_total` uses pandas' default summation: partial missing channels produce a partial total, and an all-missing row sums to zero. This matters when interpreting the inverter DC discrepancy later.


In [ ]:
# Inspect the earlier bounds-cleaning effect before deriving string features.
display(bounds_report)
display(df[string_measurements].describe().T)

df = append_features(df, string_aggregates(df))
aggregate_columns = [
    "I_string_mean", "I_string_std", "I_string_min", "I_string_max",
    "I_string_cv", "I_string_range_ratio", "I_string_cv_roll5",
    "P_string_mean", "P_string_std", "P_string_total", "P_string_cv",
]
display(df[aggregate_columns].describe().T)
display(df[[f"I_string_{i}_dev" for i in range(1, site.n_strings + 1)]].describe().T)


## 2. Per-String Operating Voltage Proxies

For each circuit, `string_voltage_proxy` calculates $R_i=1000P_i/I_i$, converting kW to W. Despite its historical `R_string_*` name, this is a **voltage proxy in volts**, not resistance. Zero current becomes a missing denominator.

The helper adds fleet mean, standard deviation, CV and standardized per-circuit proxy deviations. These support inspection of voltage disagreement; they do not introduce a separate voltage-based label in the current string rules.


In [ ]:
df = append_features(df, string_voltage_proxy(df))
R_cols = [f"R_string_{i}" for i in range(1, site.n_strings + 1)]
display(df[R_cols + ["R_string_mean", "R_string_std", "R_string_cv"]].describe().T)


## 3. POA-Normalized Features

`string_normalized` calculates $I_{norm,i}=I_i/G_{POA}$ and $P_{norm,i}=P_i/G_{POA}$, replacing zero POA with a missing denominator. Normalization reduces irradiance-driven variation when comparing a circuit with its own history.

The helper also returns normalized-power standard deviation and CV across circuits. These features do not guarantee that temperature, shading and other effects have been eliminated.

The following diagnostics use operating mask 1, POA at least 200 W/m² and inverter status in (0, 5], matching the baseline/string evaluation criteria.


In [ ]:
df = append_features(df, string_normalized(df))
evaluable_string = (
    df["Op_Mask"].eq(1)
    & df["POA_fused"].ge(POA_MIN_STRING)
    & df["InvOpStatus_Avg"].gt(0)
    & df["InvOpStatus_Avg"].le(5)
)
normalized_columns = [f"Inorm_string_{i}" for i in range(1, site.n_strings + 1)]
display(df.loc[evaluable_string, normalized_columns].describe().T)
display(df.loc[evaluable_string, ["Pnorm_string_std", "Pnorm_string_cv"]].describe().T)
display(df.loc[evaluable_string, R_cols].describe().T)


## 4. Per-String Baseline Normalization

Raw fleet CV can reflect structural asymmetry between circuits. Instead, compare each circuit with its own characteristic irradiance response. `fit_string_baselines` takes the median normalized current over evaluable observations.

For baseline $b_i$, expected current and relative deviation are

$$I_{expected,i}=b_iG_{POA},\qquad d_i=\frac{I_i-I_{expected,i}}{I_{expected,i}}.$$

A negative deviation means lower current than this circuit's expectation; a positive deviation means higher current.

The annual median is a reference fingerprint, **not a baseline verified from healthy observations**. Persistent faults may affect it. This exploratory notebook fits the selected full year, matching the annual workflow; later forecasting evaluation should fit on training data only to prevent future-data leakage. The shared helper rejects nonfinite/nonpositive baselines.


In [ ]:
INORM_BASELINE = fit_string_baselines(df, site)
baseline_report = pd.Series(
    INORM_BASELINE, name="baseline_A_m2_per_W"
).rename_axis("string").to_frame()
display(baseline_report)

df = append_features(df, apply_string_baselines(df, INORM_BASELINE))
rel_dev_cols = [f"I_string_{i}_rel_dev" for i in range(1, site.n_strings + 1)]
display(df.loc[evaluable_string, rel_dev_cols].describe(percentiles=[0.05, 0.5, 0.95]).T)


## 5. Rolling Deviations and String Fault Labels

`apply_string_baselines` produces maximum absolute relative deviation across all seven circuits, cross-circuit standard deviation and the 5-row rolling mean of the maximum (minimum 3 observations).

Ground uses all seven circuits for imbalance. For Canopy, the label helper recomputes the rolling maximum using core circuits **2–6**; channels 1 and 7 remain in engineered features and diagnostic tables. The open-circuit check uses minimum/mean current across **all seven** circuits for both sites.

`label_string` applies these ordered rules:

| Priority | Label | Rule |
|---|---|---|
| 1 | NOT_EVALUABLE | Outside the string evaluation mask |
| 2 | MISSING_DATA | Fleet mean current missing |
| 3 | OPEN_CIRCUIT_STRING | Minimum current < 10% of fleet mean during evaluable operation |
| 4 | STRING_IMBALANCE | No open circuit and the selected core-circuit rolling absolute-deviation metric > 20% |
| Default | NORMAL | No preceding condition matched |

Because the metric is absolute, imbalance can reflect positive or negative departures. A rolling average above threshold does **not** require five consecutive faulty minutes. Partial missing channels can be skipped by aggregates and are not necessarily assigned MISSING_DATA; that check tests the fleet mean.


In [ ]:
# B: Pass configured core circuits explicitly to preserve site-specific decisions.
print(f"Core circuits for imbalance: {site.core_strings}")
print(f"Open-circuit ratio: {OPEN_CIRCUIT_RATIO}; imbalance threshold: {STRING_REL_DEV_THRESHOLD}")
df["label_string"] = label_string(df, core_strings=site.core_strings)
display(label_distribution(df["label_string"]))

# Display the actual imbalance metric used by this site's label function.
core_rel_dev_cols = [f"I_string_{i}_rel_dev" for i in site.core_strings]
if set(site.core_strings) == set(range(1, site.n_strings + 1)):
    imbalance_metric = df["I_rel_dev_max_roll5"]
else:
    imbalance_metric = df[core_rel_dev_cols].abs().max(axis=1).rolling(5, min_periods=3).mean()
display(imbalance_metric.loc[evaluable_string].describe().to_frame("label_imbalance_metric"))
report_stage("String features and labels", df, checkpoint="strings")


## 6. String Fault Diagnostics

The original notebooks inspect circuit 7 deep underperformance and the circuits driving imbalance detections. These analyses are retained below. A deviation below −40% is an inspection filter, not an extra fault category.

For Canopy, circuit 7 is excluded from imbalance decisions but can still contribute to the all-circuit open-circuit rule. Dominant-circuit counts below use the configured core circuits to match the imbalance label's scope. Instantaneous attribution can differ from whichever circuit drove the rolling history.


In [ ]:
s7_mask = evaluable_string & df["I_string_7_rel_dev"].lt(-0.40)
print(f"Circuit 7 deep underperformance: {int(s7_mask.sum()):,} observations")
print(f"Equivalent sampled hours at one-minute spacing: {s7_mask.sum() / 60:.2f}")
display(df.loc[s7_mask, [
    "POA_fused", "ShuntCurrent_A_Avg_7", "I_string_7_expected",
    "I_string_7_rel_dev", "InvPAC_kW_Avg", "label_power", "label_string",
]].head(20))
display(df.loc[s7_mask, "label_power"].value_counts().to_frame("count"))
display(pd.Series(df.index[s7_mask].hour).value_counts().sort_index().to_frame("hour_count"))
display(pd.Series(df.index[s7_mask].month).value_counts().sort_index().to_frame("month_count"))

imbalance_rows = df["label_string"].eq("STRING_IMBALANCE")
core_deviations = df.loc[imbalance_rows, core_rel_dev_cols].abs().dropna(how="all")
print("Core circuit with largest instantaneous deviation during imbalance-labeled observations:")
display(core_deviations.idxmax(axis=1).value_counts().to_frame("count"))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
df.loc[PLOT_DAY, string_current_cols].plot(ax=axes[0])
axes[0].set(ylabel="Current (A)", title=f"{site.name.title()} {PLOT_DAY}: circuit comparison")
df.loc[PLOT_DAY, rel_dev_cols].plot(ax=axes[1])
axes[1].axhline(-STRING_REL_DEV_THRESHOLD, color="black", linestyle="--")
axes[1].axhline(STRING_REL_DEV_THRESHOLD, color="black", linestyle="--")
axes[1].set_ylabel("Relative deviation")
plt.tight_layout()
plt.show()


# Inverter Level Faults

Inverter analysis compares DC input, AC output and shunt-measured circuit power, retaining the original distinction between physical indicators and inverter-reported fault codes.

## 1. Inverter Policy and Engineered Features

The configured policy is passed to **both** feature calculation and labeling.

| Setting | Original policy (default here) | Revised policy |
|---|---|---|
| Power shutdown | AC power < 0.05 kW | DC input power < 1 kW |
| Status shutdown | Missing status or status <= 0 | Also status > 5 |
| Minimum DC power for efficiency faults | > 1 kW | > 5 kW |
| Reported active fault | Positive code | Nonzero, nonmissing code |

`inverter_features` generates:

- Raw efficiency $\eta=P_{AC}/P_{DC}$ and efficiency clipped to [0, 1].
- DC discrepancy $P_{string,total}-P_{DC}$, and percentage relative to inverter DC power.
- Loading ratio $P_{AC}/P_{AC,max}$.
- Any active reported fault and active fault count over the six fault-code channels.
- A 5-row rolling mean of clipped efficiency (minimum 3 observations).
- Separate power/status shutdown flags, requiring operating mask 1 and POA at least 100 W/m².

Zero DC input becomes a missing ratio denominator. Raw efficiency above 1 is preserved for inspection; clipping changes only the rolling input. Reported fault codes are diagnostics and do not directly assign the inverter label.


In [ ]:
display(pd.Series(asdict(INVERTER_POLICY), name="selected_inverter_policy").to_frame())
df = append_features(df, inverter_features(df, site, policy=INVERTER_POLICY))
inverter_columns = [
    "inv_efficiency", "inv_efficiency_clipped", "inv_efficiency_roll5",
    "DC_power_discrepancy", "DC_power_discrepancy_pct", "inv_loading_ratio",
    "inv_any_fault_reported", "inv_fault_count",
    "inv_shutdown_power_flag", "inv_shutdown_status_flag",
]
display(df[inverter_columns].describe().T)


## 2. Inverter Shutdown and Efficiency Fault Labels

`label_inverter` applies the following rules in order:

| Priority | Label | Rule |
|---|---|---|
| 1 | NOT_EVALUABLE | Operating mask = 0 |
| 2 | MISSING_DATA | AC or DC power missing |
| 3 | INVERTER_SHUTDOWN | Either shutdown flag is 1 |
| 4 | INVERTER_EFFICIENCY_FAULT | Evaluable irradiance, status in (0, 5], no shutdown, rolling clipped efficiency < 0.85, DC power above policy minimum, loading ratio > 0.20 |
| Default | NORMAL | No preceding condition matched |

Loading/status guards reduce efficiency detections during low-load/startup behavior. Shutdown flags precede label assignment, but missing-power precedence means a flagged observation can still receive MISSING_DATA.

The helper requires POA >= 100 W/m² for shutdown/efficiency detection but only tests operating mask 0 for its NOT_EVALUABLE rule. An operating-mask 1 observation with low/missing reference-cell POA can therefore reach NORMAL if no other rule matches. This notebook preserves and exposes the script behavior.


In [ ]:
# C: Apply the same policy used to calculate the inverter features.
df["label_inverter"] = label_inverter(df, policy=INVERTER_POLICY)
display(label_distribution(df["label_inverter"]))

shutdown_power = df["inv_shutdown_power_flag"].eq(1)
shutdown_status = df["inv_shutdown_status_flag"].eq(1)
shutdown_breakdown = pd.Series({
    "power_based_only": int((shutdown_power & ~shutdown_status).sum()),
    "status_based_only": int((shutdown_status & ~shutdown_power).sum()),
    "both_flags": int((shutdown_power & shutdown_status).sum()),
    "either_flag": int((shutdown_power | shutdown_status).sum()),
    "assigned_shutdown_label": int(df["label_inverter"].eq("INVERTER_SHUTDOWN").sum()),
})
display(shutdown_breakdown.to_frame("observations"))
report_stage("Inverter features and labels", df, checkpoint="inverter")


## 3. Efficiency, DC Discrepancy and Reported-Fault Cross-Checks

Inspect raw efficiency under evaluable irradiance, normal status and DC input above the policy minimum. Count efficiency values above 1 before interpreting the clipped rolling feature.

DC discrepancy can reflect wiring losses, measurement disagreement or incomplete circuit measurements; it is not automatically a wiring-fault diagnosis. Compare physics-derived shutdown/efficiency detections with reported fault codes. Agreement supports further investigation, while disagreement can reflect timing, different sensitivities or missing measurements. Neither source is treated as ground truth.


In [ ]:
evaluable_inv = df["Op_Mask"].eq(1) & df["POA_fused"].ge(100)
normal_status = df["InvOpStatus_Avg"].gt(0) & df["InvOpStatus_Avg"].le(5)
eff_mask = evaluable_inv & normal_status & df["InvPDC_kW_Avg"].gt(INVERTER_POLICY.efficiency_min_dc_kw)
display(df.loc[eff_mask, [
    "inv_efficiency", "inv_efficiency_roll5", "DC_power_discrepancy_pct", "inv_loading_ratio",
]].describe().T)
valid_efficiency = df.loc[eff_mask, "inv_efficiency"].dropna()
above_one = int(valid_efficiency.gt(1).sum())
print(f"Raw efficiency above 1: {above_one:,} of {len(valid_efficiency):,} nonmissing evaluated values")

fault_values = df.loc[:, list(FAULT_FLAG_COLUMNS)]
if INVERTER_POLICY.fault_flags_nonzero:
    active_faults = fault_values.notna() & fault_values.ne(0)
else:
    active_faults = fault_values.gt(0)
display(active_faults.sum().to_frame("active_code_observations"))
display(pd.crosstab(
    df["label_inverter"], df["inv_any_fault_reported"],
    rownames=["physics_label"], colnames=["reported_fault"], margins=True,
))

reported = df["inv_any_fault_reported"].eq(1)
physics_fault = df["label_inverter"].isin(["INVERTER_SHUTDOWN", "INVERTER_EFFICIENCY_FAULT"])
display(pd.Series({
    "physics_fault_and_reported": int((physics_fault & reported).sum()),
    "physics_fault_without_report": int((physics_fault & ~reported).sum()),
    "reported_without_physics_fault_during_evaluable_irradiance": int((reported & ~physics_fault & evaluable_inv).sum()),
}).to_frame("observations"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
valid_efficiency.hist(bins=60, ax=axes[0])
axes[0].axvline(0.85, color="red", linestyle="--", label="Efficiency floor")
axes[0].set(xlabel="Raw DC-to-AC efficiency", ylabel="Observations", title="Evaluable operation")
axes[0].legend()
df.loc[eff_mask, "DC_power_discrepancy_pct"].dropna().hist(bins=60, ax=axes[1])
axes[1].set(xlabel="DC discrepancy (%)", ylabel="Observations", title="Shunt total vs inverter DC input")
plt.tight_layout()
plt.show()


# Final Validation, Label Summaries and Export

The labels remain separate: `label_power`, `label_string` and `label_inverter`. Cross-tabulations show whether detections coincide. This notebook does not combine them into one target or create forecasting labels.

Final checks confirm that the original rows/timestamp sequence were preserved, column names remain unique and every observation has all three labels. The stage audit documents dimensionality changes. These checks establish structural consistency, not fault-label accuracy.


In [ ]:
validate_site_year(df, YEAR)
assert len(df) == original_rows, "The pipeline changed the observation count"
assert df.index.equals(original_index), "The pipeline changed timestamps or their order"
assert not df.columns.duplicated().any(), "Duplicate output column names"
label_columns = ["label_power", "label_string", "label_inverter"]
assert df[label_columns].notna().all().all(), "Missing labels in final output"

final_summary = summarize(df)
final_summary.insert(0, "year", YEAR)
final_summary.insert(0, "site", site.name)
stage_audit = pd.DataFrame(stage_records)
display(stage_audit)
display(final_summary)
display(pd.crosstab(df["label_power"], df["label_string"], margins=True))
display(pd.crosstab(df["label_power"], df["label_inverter"], margins=True))
display(df.head())


## Save the Labeled Dataset and Methodology Record

The final CSV contains retained measurements, engineered features, three labels and one TIMESTAMP index column. Companion files save label distributions, the stage audit, and a methodology record containing configuration, policy, annual string baselines and repository commit.

`OVERWRITE_FINAL = True` permits replacement of these final files, including datasets created by the batch runner. The walkthrough does not reproduce every original intermediate filename: optional checkpoints use the shared `save_stage` convention instead.

After running, use the actual tables/plots to write the selected site/year findings. Discuss sampling gaps, missing-data precedence, model assumptions, baseline fit period and heuristic thresholds alongside the observed distributions.


In [ ]:
export_paths = [final_path, summary_path, audit_path, metadata_path]
existing = [str(path) for path in export_paths if path.exists()]
if existing and not OVERWRITE_FINAL:
    raise FileExistsError(
        "Outputs already exist; set OVERWRITE_FINAL = True to replace them:\n"
        + "\n".join(existing)
    )

repo_commit = subprocess.run(
    ["git", "-C", str(repo_path), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
methodology = {
    "site": asdict(site),
    "year": YEAR,
    "repository_commit": repo_commit,
    "inverter_policy": asdict(INVERTER_POLICY),
    "string_baselines": INORM_BASELINE,
    "baseline_fit_period": "Full selected year; evaluable observations",
    "performance_rounding_decimals": 3,
    "coverage": {key: int(value) for key, value in coverage.items()},
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "exported_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
output_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(final_path, index=True, index_label="TIMESTAMP")
final_summary.to_csv(summary_path, index=False)
stage_audit.to_csv(audit_path, index=False)
metadata_path.write_text(json.dumps(methodology, indent=2), encoding="utf-8")
for path in export_paths:
    print(f"Saved: {path}")
